# MEDUSA: SFT & GRPO Training Pipeline (Colab Native)

This notebook acts as your orchestrator to fine-tune `Qwen2.5-1.5B/3B` into an Autonomous Data Engineer.
Everything executes seamlessly within Colab cells so you can monitor live training graphs and progress bars.

In [ ]:
# 1. Install dependencies
!pip install --quiet trl transformers datasets openenv-core peft huggingface_hub

# 2. Clone the Medusa Repo to get the solver scripts
!git clone https://github.com/rampluto/Medusa.git
%cd Medusa

print("Dependencies installed and repo cloned!")

## 1. Authentication
Log into your Hugging Face account to enable model pushing when training finishes.

In [ ]:
from huggingface_hub import login
# Use your WRITE token here
login("hf_YOUR_TOKEN_HERE")

## 2. Fast SFT Synthetic Dataset Generation
We dynamically generate identical data engineering pipelines including `<think>` reasoning traces into a local file.

In [ ]:
!python3 scripts/generate_sft_dataset.py --episodes 100 --out medusa_sft_100_episodes.jsonl

## 3. Dataset Loading & Smoke Testing
Because we run in Colab, we no longer need to push data to Hugging Face. We simply load the JSONL we just created!

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files="medusa_sft_100_episodes.jsonl")
print(f"Loaded {len(ds['train'])} SFT interaction steps.")

assert len(ds['train']) > 1000, "Error: Too few steps produced. Check generation script!"

## 4. Supervised Fine-Tuning (SFT) on Hugging Face Jobs
This section launches SFT on Hugging Face GPU hardware and streams logs here from Colab.

In [ ]:
import os
import re
import subprocess

# Optional: verify auth (requires HF_TOKEN in env, or prior `hf auth login`)
subprocess.run(["hf", "auth", "whoami"], check=False)

# Choose hardware flavor from `hf jobs hardware`
HF_FLAVOR = "a10g-large"   # good default for Qwen 3B LoRA
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"
HUB_MODEL_ID = "anubhavkamal/medusa-qwen-sft"
DATASET_PATH = "medusa_sft_100_episodes.jsonl"

job_cmd = [
    "hf", "jobs", "run",
    "pytorch/pytorch:2.3.1-cuda12.1-cudnn8-runtime",
    (
        "bash -lc '"
        "pip install -q -U trl transformers datasets peft accelerate huggingface_hub && "
        "git clone https://github.com/rampluto/Medusa.git && "
        "cd Medusa && "
        "python scripts/generate_sft_dataset.py --episodes 100 --out medusa_sft_100_episodes.jsonl && "
        f"python train_sft.py --dataset {DATASET_PATH} --model-id {MODEL_ID} "
        "--batch-size 2 --grad-accum 8 --max-seq-len 1024 --num-train-epochs 1 "
        "--push-to-hub "
        f"--hub-model-id {HUB_MODEL_ID}'"
    ),
    "--flavor", HF_FLAVOR,
    "--env", "HF_TOKEN=$HF_TOKEN",
    "--timeout", "12h",
    "--detach",
]

res = subprocess.run(job_cmd, capture_output=True, text=True, check=False)
print(res.stdout)
if res.returncode != 0:
    print(res.stderr)
    raise RuntimeError("Failed to submit HF Job")

# Extract job id from output
match = re.search(r"\b([a-z0-9]{8,})\b", res.stdout.lower())
JOB_ID = match.group(1) if match else None
print("Submitted JOB_ID:", JOB_ID)
if not JOB_ID:
    raise RuntimeError("Could not parse JOB_ID from hf jobs run output.")

In [ ]:
# Stream remote Hugging Face Job logs in real time
if not JOB_ID:
    raise RuntimeError("JOB_ID is missing. Run the previous cell first.")

!hf jobs logs $JOB_ID --follow

## 5. GRPO Reinforcement Learning
Tests autonomous interactions with your `openenv` Space to update behavior natively.

In [ ]:
from trl import GRPOTrainer, GRPOConfig
from openenv import SyncEnvClient as EnvironmentClient
import json
import re

def medusa_reward_function(completions, prompts):
    rewards = []
    env = EnvironmentClient("anubhavkamal/medusa-env") 
    for completion in completions:
        text = completion[0]["content"] if isinstance(completion, list) else completion
        clean_text = text.strip()
        match = re.search(r"```(?:json)?\s*(.*?)\s*```", clean_text, re.DOTALL)
        if match:
            clean_text = match.group(1).strip()
        
        try:
            payload = json.loads(clean_text)
            action = {"action": payload.get("action", "DO_NOTHING"), "params": payload.get("params", {})}
            obs, reward, terminated, truncated, info = env.step(action)
            rewards.append(float(reward))
        except Exception:
            rewards.append(-1.0)
            
    return rewards

grpo_config = GRPOConfig(
    output_dir="./medusa-grpo",
    learning_rate=5e-6,
    per_device_train_batch_size=2,
    push_to_hub=True,
    hub_model_id="anubhavkamal/medusa-qwen-grpo",
    report_to="none"
)

grpo_trainer = GRPOTrainer(model=model, reward_funcs=medusa_reward_function, args=grpo_config, train_dataset=ds['train'])

print("Starting GRPO Train...")
grpo_trainer.train()